# P68: GPU-Required Sensitivity Analyses

**Runtime: GPU (A100 or V100 recommended)**

Analyses that require re-extracting embeddings from modified images:
- §5.4.2 Image quality degradation
- §5.4.3 Test-Time Augmentation (TTA)
- §5.4.17 Lung segmentation sensitivity
- §5.5.15 Prediction stability under perturbation

Uses RAD-DINO (primary model) on a subset of test images.

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────
import subprocess, sys, os, time
from google.colab import drive
drive.mount('/content/drive')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision', 'transformers', 'pandas', 'pyarrow',
    'Pillow', 'tqdm', 'scikit-learn', 'lungmask', 'SimpleITK'],
    capture_output=True)
print('Packages installed.', flush=True)

import torch
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image, ImageFilter
from tqdm.auto import tqdm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import normalize
from sklearn.isotonic import IsotonicRegression
from transformers import AutoModel, AutoImageProcessor
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)', flush=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/P68-TB-Triage')
EMB_DIR = DRIVE_ROOT / 'data' / 'interim' / 'embeddings'
RAW = DRIVE_ROOT / 'data' / 'raw'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'tables'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# Load splits and baseline embeddings
splits = pd.read_parquet(DRIVE_ROOT / 'data' / 'processed' / 'splits.parquet')
baseline = pd.read_parquet(EMB_DIR / 'rad_dino.parquet')
emb_cols = [c for c in baseline.columns if c.startswith('emb_')]
print(f'Splits: {len(splits)}, Baseline embeddings: {len(baseline)}', flush=True)

# Load RAD-DINO model
print('Loading RAD-DINO...', flush=True)
processor = AutoImageProcessor.from_pretrained('microsoft/rad-dino', use_fast=False)
model = AutoModel.from_pretrained('microsoft/rad-dino').to(device)
model.eval()
print('Ready.', flush=True)

In [ ]:
# ── Cell 2: Helpers ──────────────────────────────────────────────────

def extract_batch(images, model, processor, device, batch_size=64):
    """Extract RAD-DINO embeddings from a list of PIL images."""
    all_emb = []
    for i in range(0, len(images), batch_size):
        batch = images[i:i+batch_size]
        inputs = processor(images=batch, return_tensors='pt')['pixel_values'].to(device)
        with torch.no_grad():
            emb = model(pixel_values=inputs).last_hidden_state[:, 0, :]
        all_emb.append(emb.cpu().float().numpy())
    return np.vstack(all_emb)


def get_test_images(splits_df, n_max=None):
    """Load test-set images with labels. Returns (images, y, patient_ids)."""
    test = splits_df[
        (splits_df['split'] == 'test') &
        (splits_df['tb_binary'].isin(['tb_positive', 'tb_negative']))
    ].copy()
    if n_max and len(test) > n_max:
        test = test.sample(n=n_max, random_state=SEED)

    images, ys, pids, valid = [], [], [], []
    for _, row in tqdm(test.iterrows(), total=len(test), desc='Loading images'):
        try:
            img = Image.open(row['file_path']).convert('RGB')
            images.append(img)
            ys.append(1 if row['tb_binary'] == 'tb_positive' else 0)
            pids.append(row['patient_id'])
            valid.append(True)
        except:
            valid.append(False)
    return images, np.array(ys), pids


def get_baseline_pipeline():
    """Get trained probe + isotonic + conformal from baseline embeddings."""
    df = baseline.copy()
    mask = df['tb_binary'].isin(['tb_positive', 'tb_negative'])
    df = df[mask]
    X = normalize(df[emb_cols].values.astype(np.float32), norm='l2')
    y = (df['tb_binary'] == 'tb_positive').astype(int).values
    sp = df['split'].values

    cal_m = sp == 'calibration'; dev_m = sp == 'dev'; test_m = sp == 'test'
    probe = LogisticRegression(C=10, max_iter=2000, solver='lbfgs', random_state=SEED)
    probe.fit(X[cal_m], y[cal_m])

    dp = probe.predict_proba(X[dev_m])[:, 1]
    iso = IsotonicRegression(out_of_bounds='clip')
    iso.fit(dp, y[dev_m])

    cal_prob = iso.predict(probe.predict_proba(X[cal_m])[:, 1])
    cal_y = y[cal_m]

    # Baseline test metrics
    test_prob = iso.predict(probe.predict_proba(X[test_m])[:, 1])
    baseline_auroc = roc_auc_score(y[test_m], probe.predict_proba(X[test_m])[:, 1])

    return probe, iso, cal_prob, cal_y, baseline_auroc


def evaluate_new_embeddings(new_X, y, probe, iso, cal_prob, cal_y, alpha=0.10):
    """Run probe + isotonic + Mondrian on new embeddings."""
    X_norm = normalize(new_X.astype(np.float32), norm='l2')
    raw_prob = probe.predict_proba(X_norm)[:, 1]
    prob = iso.predict(raw_prob)
    auroc = roc_auc_score(y, raw_prob) if len(np.unique(y)) > 1 else float('nan')

    # Mondrian conformal
    cal_sc = np.where(cal_y == 1, 1 - cal_prob, cal_prob)
    th = {}
    for c in [0, 1]:
        cs = cal_sc[cal_y == c]; n = len(cs)
        th[c] = np.quantile(cs, min(np.ceil((n+1)*(1-alpha))/n, 1.0))
    sets = []
    for p in prob:
        s = set()
        if (1-p) <= th.get(1, 1.0): s.add(1)
        if p <= th.get(0, 1.0): s.add(0)
        sets.append(s)

    cov = [int(yi in s) for yi, s in zip(y, sets)]
    tb_m = y == 1
    tb_cov = np.mean([cov[i] for i in range(len(cov)) if tb_m[i]]) if tb_m.any() else float('nan')
    singleton = np.mean([len(s) == 1 for s in sets])
    return {'auroc': round(auroc, 4), 'tb_cov': round(tb_cov, 4), 'singleton': round(singleton, 4)}


probe, iso, cal_prob, cal_y, baseline_auroc = get_baseline_pipeline()
print(f'Baseline AUROC: {baseline_auroc:.4f}')

In [ ]:
# ── Cell 3: Copy test images to local disk ───────────────────────────
# Same as before — Drive is slow for random reads

import shutil
LOCAL_IMG = Path('/content/local_images')

existing = sum(1 for _ in LOCAL_IMG.rglob('*.png')) + sum(1 for _ in LOCAL_IMG.rglob('*.jpg')) if LOCAL_IMG.exists() else 0
if existing >= len(splits) * 0.9:
    print(f'Already cached ({existing} files). Skipping copy.', flush=True)
else:
    for name, src_dir in [('shenzhen_montgomery', RAW/'shenzhen_montgomery'),
                           ('tbx11k', RAW/'tbx11k'),
                           ('chexpert_subset', RAW/'chexpert_subset'),
                           ('mendeley_pakistan', RAW/'mendeley_pakistan')]:
        dst = LOCAL_IMG / name
        if dst.exists() and any(dst.rglob('*.png')) or any(dst.rglob('*.jpg')):
            print(f'{name}: cached', flush=True); continue
        if not src_dir.exists():
            print(f'{name}: source missing', flush=True); continue
        print(f'{name}: copying...', flush=True)
        os.system(f'cp -r "{src_dir}" "{dst}"')
        print(f'  done', flush=True)

# Remap splits paths to local
def remap(row):
    p = Path(row['file_path'])
    for ds_map in [('shenzhen', 'shenzhen_montgomery'), ('montgomery', 'shenzhen_montgomery'),
                   ('tbx11k', 'tbx11k'), ('chexpert', 'chexpert_subset'), ('pakistan', 'mendeley_pakistan')]:
        if row['dataset'] == ds_map[0] or row['dataset'] == ds_map[0]:
            local = LOCAL_IMG / ds_map[1]
            if local.exists():
                matches = list(local.rglob(p.name))
                if matches: return str(matches[0])
    return row['file_path']

splits['file_path'] = splits.apply(remap, axis=1)
n_local = splits['file_path'].str.startswith('/content/local').sum()
print(f'Remapped: {n_local} local, {len(splits)-n_local} Drive', flush=True)

In [ ]:
# ── Cell 4: §5.4.2 IMAGE QUALITY DEGRADATION ─────────────────────────
# Re-extract embeddings at different degradation levels.
# Uses full test set for each degradation.

print('§5.4.2 IMAGE QUALITY DEGRADATION')
print('=' * 60, flush=True)

test_images, test_y, test_pids = get_test_images(splits)
print(f'Loaded {len(test_images)} test images.', flush=True)

# Baseline (no degradation)
print('\nExtracting baseline...', flush=True)
baseline_emb = extract_batch(test_images, model, processor, device)
base_metrics = evaluate_new_embeddings(baseline_emb, test_y, probe, iso, cal_prob, cal_y)
print(f'  Baseline: {base_metrics}', flush=True)

results = [{'degradation': 'none', 'level': 'original', **base_metrics}]

# Resolution downsampling
for res in [512, 256, 128]:
    print(f'\n  Resolution {res}x{res}...', flush=True)
    degraded = [img.resize((res, res), Image.BILINEAR).resize(img.size, Image.BILINEAR) for img in test_images]
    emb = extract_batch(degraded, model, processor, device)
    m = evaluate_new_embeddings(emb, test_y, probe, iso, cal_prob, cal_y)
    results.append({'degradation': 'resolution', 'level': f'{res}x{res}', **m})
    print(f'    {m}', flush=True)
    del degraded, emb

# JPEG compression
import io
for quality in [90, 70, 50, 30]:
    print(f'\n  JPEG quality {quality}...', flush=True)
    degraded = []
    for img in test_images:
        buf = io.BytesIO()
        img.save(buf, format='JPEG', quality=quality)
        buf.seek(0)
        degraded.append(Image.open(buf).convert('RGB'))
    emb = extract_batch(degraded, model, processor, device)
    m = evaluate_new_embeddings(emb, test_y, probe, iso, cal_prob, cal_y)
    results.append({'degradation': 'jpeg', 'level': f'q{quality}', **m})
    print(f'    {m}', flush=True)
    del degraded, emb

# Gaussian noise
for sigma in [0.01, 0.05, 0.10]:
    print(f'\n  Gaussian noise sigma={sigma}...', flush=True)
    degraded = []
    for img in test_images:
        arr = np.array(img).astype(np.float32) / 255.0
        noisy = arr + np.random.normal(0, sigma, arr.shape)
        noisy = np.clip(noisy * 255, 0, 255).astype(np.uint8)
        degraded.append(Image.fromarray(noisy))
    emb = extract_batch(degraded, model, processor, device)
    m = evaluate_new_embeddings(emb, test_y, probe, iso, cal_prob, cal_y)
    results.append({'degradation': 'noise', 'level': f'sigma={sigma}', **m})
    print(f'    {m}', flush=True)
    del degraded, emb

# Brightness/contrast jitter ±20%
for factor_name, factor in [('bright+20%', 1.2), ('bright-20%', 0.8)]:
    print(f'\n  {factor_name}...', flush=True)
    from PIL import ImageEnhance
    degraded = [ImageEnhance.Brightness(img).enhance(factor) for img in test_images]
    emb = extract_batch(degraded, model, processor, device)
    m = evaluate_new_embeddings(emb, test_y, probe, iso, cal_prob, cal_y)
    results.append({'degradation': 'brightness', 'level': factor_name, **m})
    print(f'    {m}', flush=True)
    del degraded, emb

deg_df = pd.DataFrame(results)
deg_df.to_csv(RESULTS_DIR / 'image_degradation.csv', index=False)
print(f'\nSaved: image_degradation.csv')
print(deg_df.to_string(index=False))

In [ ]:
# ── Cell 5: §5.4.3 TEST-TIME AUGMENTATION (TTA) ──────────────────────

print('§5.4.3 TEST-TIME AUGMENTATION (TTA)')
print('=' * 60, flush=True)

# K=5 augmentations: horizontal flip, ±5° rotation, ±5% crop
augmentations = [
    ('original', lambda img: img),
    ('hflip', lambda img: TF.hflip(img)),
    ('rot+5', lambda img: TF.rotate(img, 5, fill=0)),
    ('rot-5', lambda img: TF.rotate(img, -5, fill=0)),
    ('crop95', lambda img: TF.center_crop(img, [int(s*0.95) for s in img.size[::-1]])),
]

# Extract embeddings for each augmentation
aug_embeddings = {}
for aug_name, aug_fn in augmentations:
    print(f'  {aug_name}...', flush=True)
    augmented = [aug_fn(img) for img in test_images]
    aug_embeddings[aug_name] = extract_batch(augmented, model, processor, device)
    del augmented

# Average embeddings across augmentations
tta_emb = np.mean([aug_embeddings[name] for name in aug_embeddings], axis=0)

# Compare
no_tta = evaluate_new_embeddings(aug_embeddings['original'], test_y, probe, iso, cal_prob, cal_y)
with_tta = evaluate_new_embeddings(tta_emb, test_y, probe, iso, cal_prob, cal_y)

print(f'\n  Without TTA: {no_tta}')
print(f'  With TTA:    {with_tta}')
print(f'  AUROC delta: {with_tta["auroc"] - no_tta["auroc"]:+.4f}')

tta_df = pd.DataFrame([{'method': 'no_tta', **no_tta}, {'method': 'tta_k5', **with_tta}])
tta_df.to_csv(RESULTS_DIR / 'tta_results.csv', index=False)
print(f'\nSaved: tta_results.csv')

del aug_embeddings, tta_emb

In [ ]:
# ── Cell 6: §5.4.17 LUNG SEGMENTATION SENSITIVITY ────────────────────

print('§5.4.17 LUNG SEGMENTATION SENSITIVITY')
print('=' * 60, flush=True)

# Use a simple approach: threshold-based lung region approximation
# (Full U-Net segmentation requires SimpleITK + lungmask which can be fragile)

try:
    from lungmask import LMInferer
    import SimpleITK as sitk

    inferer = LMInferer(modelname='R231')
    use_lungmask = True
    print('  Using lungmask R231 model.', flush=True)
except Exception as e:
    print(f'  lungmask failed ({e}). Using intensity-based approximation.', flush=True)
    use_lungmask = False

# Process subset for speed (lung segmentation is slow)
n_seg = min(1000, len(test_images))
rng = np.random.RandomState(SEED)
seg_idx = rng.choice(len(test_images), n_seg, replace=False)
seg_images = [test_images[i] for i in seg_idx]
seg_y = test_y[seg_idx]

if use_lungmask:
    # Apply lungmask to each image
    lung_masked = []
    for i, img in enumerate(tqdm(seg_images, desc='Segmenting')):
        try:
            arr = np.array(img.convert('L'))
            sitk_img = sitk.GetImageFromArray(arr[np.newaxis, :, :])
            mask = inferer.apply(sitk_img)
            mask_2d = (mask[0] > 0).astype(np.uint8)
            # Zero out non-lung pixels
            rgb_arr = np.array(img)
            rgb_arr[mask_2d == 0] = 0
            lung_masked.append(Image.fromarray(rgb_arr))
        except:
            lung_masked.append(img)  # fallback to original
else:
    # Simple intensity-based: keep pixels between 20th-80th percentile
    lung_masked = []
    for img in tqdm(seg_images, desc='Masking'):
        arr = np.array(img.convert('L')).astype(float)
        p20, p80 = np.percentile(arr, 20), np.percentile(arr, 80)
        mask = ((arr > p20) & (arr < p80)).astype(np.uint8)
        rgb_arr = np.array(img)
        rgb_arr[mask == 0] = 0
        lung_masked.append(Image.fromarray(rgb_arr))

# Extract embeddings for whole-image and lung-only
print('  Extracting whole-image embeddings...', flush=True)
whole_emb = extract_batch(seg_images, model, processor, device)
print('  Extracting lung-only embeddings...', flush=True)
lung_emb = extract_batch(lung_masked, model, processor, device)

whole_m = evaluate_new_embeddings(whole_emb, seg_y, probe, iso, cal_prob, cal_y)
lung_m = evaluate_new_embeddings(lung_emb, seg_y, probe, iso, cal_prob, cal_y)

print(f'\n  Whole image: {whole_m}')
print(f'  Lung only:   {lung_m}')
delta_auroc = lung_m['auroc'] - whole_m['auroc']
print(f'  AUROC delta: {delta_auroc:+.4f}')
if delta_auroc >= 0.02:
    print('  -> Lung-only IMPROVES: whole-image may rely on non-lung shortcuts')
elif delta_auroc <= -0.02:
    print('  -> Lung-only DEGRADES: contextual info (heart, body) carries TB signal')
else:
    print('  -> Minimal difference: lung masking has little effect')

seg_df = pd.DataFrame([{'method': 'whole_image', 'n': n_seg, **whole_m},
                         {'method': 'lung_only', 'n': n_seg, **lung_m}])
seg_df.to_csv(RESULTS_DIR / 'lung_segmentation.csv', index=False)
print(f'\nSaved: lung_segmentation.csv')

del lung_masked, whole_emb, lung_emb

In [ ]:
# ── Cell 7: §5.5.15 PREDICTION STABILITY UNDER PERTURBATION ──────────

print('§5.5.15 PREDICTION STABILITY UNDER PERTURBATION')
print('=' * 60, flush=True)

# 200 random test images × 10 perturbations each
n_stab = 200
n_perturb = 10
rng = np.random.RandomState(SEED)
stab_idx = rng.choice(len(test_images), n_stab, replace=False)
stab_images = [test_images[i] for i in stab_idx]
stab_y = test_y[stab_idx]

# Get original prediction sets
orig_emb = extract_batch(stab_images, model, processor, device)
orig_X = normalize(orig_emb.astype(np.float32), norm='l2')
orig_prob = iso.predict(probe.predict_proba(orig_X)[:, 1])

# Mondrian thresholds from calibration
cal_sc = np.where(cal_y == 1, 1 - cal_prob, cal_prob)
th = {}
for c in [0, 1]:
    cs = cal_sc[cal_y == c]; n = len(cs)
    th[c] = np.quantile(cs, min(np.ceil((n+1)*0.9)/n, 1.0))

def get_set(p):
    s = set()
    if (1-p) <= th.get(1, 1.0): s.add(1)
    if p <= th.get(0, 1.0): s.add(0)
    return frozenset(s)

orig_sets = [get_set(p) for p in orig_prob]

# Perturbations
stability_scores = []
print(f'  Testing {n_stab} images × {n_perturb} perturbations...', flush=True)

for pert_i in tqdm(range(n_perturb), desc='Perturbations'):
    perturbed = []
    for img in stab_images:
        # Random combination of small perturbations
        p_img = img.copy()
        # ±1° rotation
        angle = rng.uniform(-1, 1)
        p_img = TF.rotate(p_img, angle, fill=0)
        # ±2% crop
        w, h = p_img.size
        crop_frac = rng.uniform(0.98, 1.0)
        new_w, new_h = int(w * crop_frac), int(h * crop_frac)
        p_img = TF.center_crop(p_img, [new_h, new_w])
        p_img = p_img.resize((w, h), Image.BILINEAR)
        # ±5% brightness
        from PIL import ImageEnhance
        bright_factor = rng.uniform(0.95, 1.05)
        p_img = ImageEnhance.Brightness(p_img).enhance(bright_factor)
        # Tiny noise
        arr = np.array(p_img).astype(float)
        arr += rng.normal(0, 1.3, arr.shape)  # sigma=0.005 * 255
        arr = np.clip(arr, 0, 255).astype(np.uint8)
        perturbed.append(Image.fromarray(arr))

    pert_emb = extract_batch(perturbed, model, processor, device)
    pert_X = normalize(pert_emb.astype(np.float32), norm='l2')
    pert_prob = iso.predict(probe.predict_proba(pert_X)[:, 1])
    pert_sets = [get_set(p) for p in pert_prob]

    # Compare to original
    matches = [int(o == p) for o, p in zip(orig_sets, pert_sets)]
    stability_scores.append(matches)
    del perturbed, pert_emb

# Per-image stability score = fraction of perturbations producing same set
stability_matrix = np.array(stability_scores).T  # [n_images, n_perturbations]
per_image_stability = stability_matrix.mean(axis=1)

print(f'\n  Mean stability score: {per_image_stability.mean():.4f}')
print(f'  Median: {np.median(per_image_stability):.4f}')
print(f'  Images with stability < 0.80: {(per_image_stability < 0.80).mean():.1%}')

# Stratify by original prediction set type
for set_type, set_val in [('singleton {TB}', frozenset({1})),
                           ('singleton {nonTB}', frozenset({0})),
                           ('uncertain {TB,nonTB}', frozenset({0, 1})),
                           ('empty {}', frozenset())]:
    mask = [s == set_val for s in orig_sets]
    if any(mask):
        sub_stab = per_image_stability[mask]
        print(f'  {set_type:25s}: n={sum(mask):>5}, stability={sub_stab.mean():.4f}')

stab_df = pd.DataFrame({'patient_idx': stab_idx, 'stability_score': per_image_stability,
                          'y_true': stab_y, 'orig_set': [str(s) for s in orig_sets]})
stab_df.to_csv(RESULTS_DIR / 'perturbation_stability.csv', index=False)
print(f'\nSaved: perturbation_stability.csv')

In [ ]:
# ── Cell 8: Verification ──────────────────────────────────────────────

print('GPU sensitivity analyses complete.')
print('\nResults saved to Drive:')
for f in ['image_degradation.csv', 'tta_results.csv', 'lung_segmentation.csv', 'perturbation_stability.csv']:
    p = RESULTS_DIR / f
    if p.exists():
        print(f'  {f:40s}  {p.stat().st_size/1e3:.1f} KB')
    else:
        print(f'  {f:40s}  MISSING')

print('\nCopy these to your local results/tables/ directory.')

# Cleanup
del model, processor
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()